In [2]:
import torch
import numpy as np
from board import Board
from torch import nn

gs = Board()

matrix = gs.createMatrix()
print(matrix.shape)

(13, 8, 8)


In [3]:
boardTensor = torch.from_numpy(matrix)
print(boardTensor)

tensor([[[0, 0, 0, 0, 0, 0, 0, 0],
         [1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0]],

        [[1, 0, 0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 1, 0, 0, 0, 0, 1, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 1, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0

In [4]:
import torch

# This logic checks what hardware is available
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 13, kernel_size = 3, stride = 1, out_channels = 64, padding = 1)
        self.conv2 = nn.Conv2d(in_channels = 64, kernel_size = 3, stride = 1, out_channels = 128, padding = 1)
        self.stack = nn.Sequential(self.conv1, nn.ReLU(), self.conv2, nn.ReLU(), nn.Flatten(), nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, x):
        logits = self.stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (conv1): Conv2d(13, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (stack): Sequential(
    (0): Conv2d(13, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=8192, out_features=256, bias=True)
    (6): ReLU()
    (7): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [6]:
floatTensor = boardTensor.float()

floatTensor = torch.unsqueeze(floatTensor, 0)

output = model(floatTensor)
print(output)

tensor([[0.0373]], grad_fn=<AddmmBackward0>)


In [ ]:
def steps(gs):
    moves = gs.get_all_legal_moves()
    scores = []
    storedBitboards = {}
    for start, target in moves:
        currentBoards = gs.createCurrentBoards(gs.getBoards())
        gs.make_move(start, target)
        matrix = gs.createMatrix()
        tens = torch.from_numpy(matrix)
        tens = tens.float()
        tens = torch.unsqueeze(tens, 0)
        with torch.no_grad():
            score = model(tens)
        
        scores.append(score.item())
        gs.setBoards(currentBoards)
        gs.setBoardArray()
        gs.switchColor()

    if (gs.pieceColor == 1):
        bestMove = np.argmax(scores)
    else:
        bestMove = np.argmin(scores)
    return moves[bestMove]
    

In [11]:
print(f"Number of moves evaluated: {len(scores)}")
print(f"Scores: {scores}")
if gs.pieceColor == 1:
    print("It's white's turn. Looking for HIGHEST score.")
    best_move_index = np.argmax(scores)
else:
    print("It's black's t urn. Looking for LOWEST score.")
    best_move_index = np.argmin(scores)
chosen_move = moves[best_move_index]
print(f"The AI chooses move index {best_move_index}: {chosen_move}")

Number of moves evaluated: 20
Scores: [0.03385230898857117, 0.033236052840948105, 0.03333011269569397, 0.03218863531947136, 0.03334644436836243, 0.033942341804504395, 0.032221220433712006, 0.03187881410121918, 0.03263381868600845, 0.03481258451938629, 0.035639118403196335, 0.03323695436120033, 0.0309909675270319, 0.030382715165615082, 0.03484516218304634, 0.03369239717721939, 0.03441969305276871, 0.036657266318798065, 0.03554534912109375, 0.03547535091638565]
It's white's turn. Looking for HIGHEST score.
The AI chooses move index 17: (14, 30)


In [12]:
board = Board()
gameMatrices = []
in_progress = True
moveCount = 0

while (in_progress):
    bestMove = steps(board)
    gameMatrices.append(board.createMatrix())
    board.make_move(bestMove[0], bestMove[1])
    if (board.check_game_status() == "CHECKMATE" or board.check_game_status() == "DRAW" or moveCount > 200):
        in_progress = False
    print("Start: " + bestMove[0] + ", End: " + bestMove[1])
    moveCount += 1


TypeError: steps() takes 0 positional arguments but 1 was given